In [1]:
import os
import pandas as pd
import scipy
import cobra
from cobra.io import load_matlab_model
from cobra.io import read_sbml_model
from cobra.flux_analysis import gapfill
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model
from cobra.flux_analysis import fastcc
from cobra.flux_analysis import flux_variability_analysis

# Adding ApoE gene and associated mets/reaction using Recon3D model

In [2]:
# read the models
iMiceBrain = load_matlab_model('/Users/eso1993/Desktop/iMiceBrain_consensus.mat')
iMM1865 = load_matlab_model('/Users/eso1993/Library/CloudStorage/Box-Box/PFOS_Project/mouse_model_iMM1865/iMM1865_updated.mat') 
recon3d = load_matlab_model('/Users/eso1993/Desktop/Recon3D_301.mat')

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x
No defined compartments in model Recon3D. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


In [3]:
# identify the ApoE related mets in recon3d
mets_hint = ["apoe", "density", "triglyc"]

matches = [
    met for met in iMM1865.metabolites
    if any(hint in met.name.lower() for hint in mets_hint)
]

# Print matched metabolite IDs and names
for m in matches:
    print(f"{m.id}: {m.name}")

tag_hs[c]: Triglyceride
tag_hs[e]: Triglyceride
HC00009[e]: ApoE
idl_hs[e]: Intermediate Density Lipoprotein
ldl_hs[e]: Low Density Lipoprotein
hdl_hs[e]: High Density Lipoprotein


In [4]:
# get the chemical equations for each reaction of the aformentioned reactions list 
for met in matches:
    print(f"\nMetabolite: {met.id} - {met.name}")
    for rxn in met.reactions:
        print(f"  Reaction ID: {rxn.id}")
        print(f"  Equation: {rxn.reaction}")
        print(f"  GPR: {rxn.gene_reaction_rule}")


Metabolite: tag_hs[c] - Triglyceride
  Reaction ID: SK_tag_hs_c
  Equation: tag_hs[c] <=> 
  GPR: 
  Reaction ID: DGAT
  Equation: Rtotal3coa[c] + dag_hs[c] --> coa[c] + tag_hs[c]
  GPR: 13350 or 67800
  Reaction ID: TAGt
  Equation: tag_hs[e] <=> tag_hs[c]
  GPR: 
  Reaction ID: LPS
  Equation: h2o[c] + tag_hs[c] --> Rtotal3[c] + dag_hs[c] + h[c]
  GPR: 16956 or 15450 or 12613 or 116939

Metabolite: tag_hs[e] - Triglyceride
  Reaction ID: EX_tag_hs_e
  Equation: tag_hs[e] <=> 
  GPR: 
  Reaction ID: TAGt
  Equation: tag_hs[e] <=> tag_hs[c]
  GPR: 
  Reaction ID: IDL_HSSYN
  Equation: 0.5 HC00005[e] + 0.5 HC00009[e] + 4.0 chsterol[e] + 4.0 tag_hs[e] --> idl_hs[e]
  GPR: 
  Reaction ID: LDL_HSSYN
  Equation: 2.0 HC00005[e] + 5.0 chsterol[e] + 2.0 pchol_hs[e] + tag_hs[e] --> ldl_hs[e]
  GPR: 
  Reaction ID: HDL_HSSYN
  Equation: HC00004[e] + HC00006[e] + HC00007[e] + HC00008[e] + HC00009[e] + 2.0 chsterol[e] + 2.0 pchol_hs[e] + tag_hs[e] --> hdl_hs[e]
  GPR: 

Metabolite: HC00009[e] - A

In [5]:
# add these reactions and then make changes in it accordingly
reaction_ids = [
    "LPS", "TAGt", "DGAT", "SK_tag_hs_c",
    "IDL_HSSYN", "LDL_HSSYN", "HDL_HSSYN", "EX_tag_hs_e",
    "EX_HC00009_e", "IDL_HSDEG", "HDL_HSDEG",
    "EX_idl_hs_e", "LDL_HSDEG", "EX_ldl_hs_e",
    "EX_hdl_hs_e"
]

hdl_apoe_reactions = [
    iMM1865.reactions.get_by_id(rxn_id)
    for rxn_id in reaction_ids
    if rxn_id in iMM1865.reactions
]

iMiceBrain.add_reactions(hdl_apoe_reactions)

Ignoring reaction 'LPS' since it already exists.
Ignoring reaction 'DGAT' since it already exists.
Ignoring reaction 'SK_tag_hs_c' since it already exists.


In [6]:
# check disconnencted metabolites 
disconnected_mets = [met for met in iMiceBrain.metabolites if len(met.reactions) <= 1]

print(f"Number of potentially disconnected metabolites: {len(disconnected_mets)}")
for met in disconnected_mets:
    print(met.id, met.name)

Number of potentially disconnected metabolites: 2268
ksi_deg17[l] Keratan Sulfate I, Degradation Product 17
ksi_deg18[l] Keratan Sulfate I, Degradation Product 18
glyc__S[e] (S)-Glycerate
glyc__S[c] (S)-Glycerate
xolest182_hs[e] 1-Linoleoyl-Cholesterol, Cholesterol-Ester (18:2, Delta 9, 12)
xolest182_hs[c] 1-Linoleoyl-Cholesterol, Cholesterol-Ester (18:2, Delta 9, 12)
docohepcoa[m] 2,4,7,10,13,16,19-Docosaheptenoyl Coenzyme A
gdpfuc[c] Guanosine-5'-Diphosphate-L-Fucose
fucacngal14acglcgalgluside_hs[c] Iv3-A-Neuac,Iii3-A-Fuc-Nlc4Cer
acngal14acglcgalgluside_hs[c] Alpha-N-Acetylneuraminyl-2,3-Beta-D-Galactosyl-1,4-N-Acetyl-Beta-D-Glucosaminyl-1,3-Beta-D-Galactosyl-1,4-D-Glucosylceramide
ksi_deg41[l] Keratan Sulfate I, Degradation Product 41
cs_c_pre2[g] Chondroitin Sulfate C (GalNac6S-Glca), Precursor 2
cs_c_d_e_pre1[g] Chondroitin Sulfate C (GalNac6S-Glca) And D (GlcNac6S-Glca2S), Precursor 1
ksi_deg24[l] Keratan Sulfate I, Degradation Product 24
ksi_deg25[l] Keratan Sulfate I, Degradati

In [7]:
# check blocked reactions 
block = cobra.flux_analysis.find_blocked_reactions(iMiceBrain)
len(block)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmped612imw.lp
Reading time = 0.02 seconds
: 5379 rows, 12063 columns, 52669 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpiih1rths.lp
Reading time = 0.02 seconds
: 5379 rows, 12063 columns, 52669 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp68ogam9h.lp
Reading time = 0.02 seconds
: 5379 rows, 12063 columns, 52669 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 

10

In [8]:
# check these reactions and associated equation
for rxn_id in block:
    rxn = iMM1865.reactions.get_by_id(rxn_id)
    print(rxn.id, rxn.reaction)

IDL_HSSYN 0.5 HC00005[e] + 0.5 HC00009[e] + 4.0 chsterol[e] + 4.0 tag_hs[e] --> idl_hs[e]
LDL_HSSYN 2.0 HC00005[e] + 5.0 chsterol[e] + 2.0 pchol_hs[e] + tag_hs[e] --> ldl_hs[e]
HDL_HSSYN HC00004[e] + HC00006[e] + HC00007[e] + HC00008[e] + HC00009[e] + 2.0 chsterol[e] + 2.0 pchol_hs[e] + tag_hs[e] --> hdl_hs[e]
EX_HC00009_e HC00009[e] <=> 
IDL_HSDEG 4.0 h2o[e] + idl_hs[e] --> 0.5 HC00005[e] + 0.5 HC00009[e] + 4.0 Rtotal2[e] + 4.0 Rtotal3[e] + 4.0 Rtotal[e] + 4.0 chsterol[e] + 4.0 glyc[e]
HDL_HSDEG h2o[e] + hdl_hs[e] --> HC00004[e] + HC00006[e] + HC00007[e] + HC00008[e] + HC00009[e] + Rtotal2[e] + Rtotal3[e] + Rtotal[e] + 2.0 chsterol[e] + glyc[e] + 2.0 pchol_hs[e]
EX_idl_hs_e idl_hs[e] <=> 
LDL_HSDEG h2o[e] + ldl_hs[e] --> 2.0 HC00005[e] + Rtotal2[e] + Rtotal3[e] + Rtotal[e] + 5.0 chsterol[e] + glyc[e] + 2.0 pchol_hs[e]
EX_ldl_hs_e ldl_hs[e] <=> 
EX_hdl_hs_e hdl_hs[e] <=> 


In [9]:
# perform gap filling first before doing any further steps 
iMiceBrain.solver = 'gurobi'
gapfill_solutions = {}

for rxn_id in block:
    rxn = iMiceBrain.reactions.get_by_id(rxn_id)
    with iMiceBrain:
        iMiceBrain.objective = rxn
        try:
            solution = gapfill(iMiceBrain, iMM1865, iterations=1, 
                               exchange_reactions=True, demand_reactions=True)
            gapfill_solutions[rxn_id] = solution
            print(f"Gapfilling successful for {rxn_id}")
        except Exception as e:
            print(f"Gapfilling failed for {rxn_id}: {e}")

Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp1scelln0.lp
Reading time = 0.02 seconds
: 5378 rows, 12062 columns, 52662 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp4zil0vuz.lp
Reading time = 0.03 seconds
: 5839 rows, 21224 columns, 80812 nonzeros
Gapfilling successful for IDL_HSSYN
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp9n3_rmij.lp
Reading time = 0.02 seconds
: 5378 rows, 12062 columns, 52662 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp_lr_qyju.lp
Reading time = 0.03 seconds
: 5839 rows, 21224 columns, 80812 nonzeros
Gapfilling successful for LDL_HSSYN
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmphbllo0sf.lp
Reading time = 0.02 seconds
: 5378 rows, 12062 columns, 52662 nonzeros
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp2e8kdhar.lp
Rea

In [10]:
# add the needed reactions as identified by the gap filling 
for rxn_id, solutions in gapfill_solutions.items():
    for rxn in solutions[0]:  # only apply first solution per target reaction
        iMiceBrain.add_reactions([rxn])

Ignoring reaction 'EX_Rtotal3_e' since it already exists.
Ignoring reaction 'EX_Rtotal3_e' since it already exists.
Ignoring reaction 'DM_Rtotal3[e]' since it already exists.
Ignoring reaction 'DM_Rtotal3[e]' since it already exists.
Ignoring reaction 'DM_Rtotal3[e]' since it already exists.
Ignoring reaction 'EX_HC00005_e' since it already exists.


In [11]:
# remove disconnected metabolites and associated reactions 
iMiceBrain.remove_metabolites(disconnected_mets)

In [12]:
len(iMiceBrain.genes), len(iMiceBrain.reactions), len(iMiceBrain.metabolites)

(1855, 6038, 3110)

In [13]:
# check blocked reactions
block_test = cobra.flux_analysis.find_blocked_reactions(iMiceBrain)
len(block_test)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmppdnb47w5.lp
Reading time = 0.02 seconds
: 3111 rows, 12077 columns, 52683 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpwag4elvk.lp
Reading time = 0.02 seconds
: 3111 rows, 12077 columns, 52683 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmp4nagv4i5.lp
Reading time = 0.02 seconds
: 3111 rows, 12077 columns, 52683 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 

0

In [14]:
# now check duplicated metabolites 
from collections import Counter

# Get all metabolite IDs from the model
met_ids = [met.id for met in iMiceBrain.metabolites]

# Count how many times each ID appears
met_id_counts = Counter(met_ids)

# Filter for duplicates
duplicates = {met_id: count for met_id, count in met_id_counts.items() if count > 1}

# Display duplicates
if duplicates:
    print("Duplicated metabolite IDs found:")
    for met_id, count in duplicates.items():
        print(f"{met_id}: {count} times")
else:
    print("No duplicated metabolite IDs found.")

No duplicated metabolite IDs found.


In [15]:
# now add the reaction for APOE from recon3d
iMiceBrain.add_reactions([recon3d.reactions.get_by_id("r1112")])
len(iMiceBrain.genes), len(iMiceBrain.reactions), len(iMiceBrain.metabolites)

(1856, 6039, 3130)

In [16]:
# chnage the gpr and remove the associated gene id for r1112
iMiceBrain.reactions.get_by_id("r1112").gene_reaction_rule = "11816"
old_id = "348.1"
if old_id in iMiceBrain.genes:
    g_old = iMiceBrain.genes.get_by_id(old_id)
    if len(g_old.reactions) == 0:
        iMiceBrain.genes.remove(g_old)

In [17]:
iMiceBrain.reactions.get_by_id("r1112").reaction

'39.0 ala_L[c] + 34.0 arg_L[c] + asn_L[c] + 11.0 asp_L[c] + 1268.0 atp[c] + 2.0 cys_L[c] + 32.0 gln_L[c] + 40.0 glu_L[c] + 18.0 gly[c] + 1268.0 h2o[c] + 2.0 his_L[c] + 2.0 ile_L[c] + 41.0 leu_L[c] + 13.0 lys_L[c] + 8.0 met_L[c] + 4.0 phe_L[c] + 8.0 pro_L[c] + 14.0 ser_L[c] + 12.0 thr_L[c] + 8.0 trp_L[c] + 4.0 tyr_L[c] + 24.0 val_L[c] --> HC00009[c] + 951.0 adp[c] + 317.0 amp[c] + 951.0 pi[c] + 317.0 ppi[c]'

In [18]:
from cobra.core.dictlist import DictList

def merge_metabolite(model, old_id, new_id, merge_notes=True, merge_annotation=True):
    """Redirect all stoichiometry from old_id to new_id, then remove old_id."""
    if old_id not in model.metabolites:
        return [], f"{old_id} not in model; skipped."
    if new_id not in model.metabolites:
        return [], f"{new_id} not in model; skipped (target missing)."

    old = model.metabolites.get_by_id(old_id)
    new = model.metabolites.get_by_id(new_id)

    changed_rxns = []
    # iterate over a copy because rxn set mutates as we edit
    for rxn in list(old.reactions):
        coeff = rxn.metabolites[old]
        # remove old, add new with same coefficient (combine=True handles if new already present)
        rxn.add_metabolites({old: -coeff, new: coeff}, combine=True)
        changed_rxns.append(rxn.id)

    # optionally merge metadata
    if merge_notes and getattr(old, "notes", None):
        try:
            new.notes.update(old.notes)
        except Exception:
            pass
    if merge_annotation and getattr(old, "annotation", None):
        try:
            new.annotation.update(old.annotation)
        except Exception:
            pass
    if getattr(new, "name", None) in (None, "", "unknown") and getattr(old, "name", None):
        new.name = old.name

    # finally remove the old metabolite from the model
    model.metabolites.remove(old)
    return changed_rxns, f"Merged {old_id} -> {new_id}"

# --- mapping: single underscore -> double underscore (cytosol) ---
pairs_c = {
    "ala_L[c]":"ala__L[c]","arg_L[c]":"arg__L[c]","asn_L[c]":"asn__L[c]",
    "asp_L[c]":"asp__L[c]","cys_L[c]":"cys__L[c]","gln_L[c]":"gln__L[c]",
    "glu_L[c]":"glu__L[c]","his_L[c]":"his__L[c]","ile_L[c]":"ile__L[c]",
    "leu_L[c]":"leu__L[c]","lys_L[c]":"lys__L[c]","met_L[c]":"met__L[c]",
    "phe_L[c]":"phe__L[c]","pro_L[c]":"pro__L[c]","ser_L[c]":"ser__L[c]",
    "thr_L[c]":"thr__L[c]","trp_L[c]":"trp__L[c]","tyr_L[c]":"tyr__L[c]",
    "val_L[c]":"val__L[c]"
}

# run the merges
summary = {}
for old_id, new_id in pairs_c.items():
    changed, msg = merge_metabolite(iMiceBrain, old_id, new_id)
    summary[old_id] = {"into": new_id, "changed_reactions": changed, "message": msg}

# quick report
merged = [k for k,v in summary.items() if v["changed_reactions"]]
print(f"Merged {len(merged)} metabolites:")
for k in merged:
    print(f"  {k} -> {summary[k]['into']} (touched {len(summary[k]['changed_reactions'])} rxns)")


Merged 19 metabolites:
  ala_L[c] -> ala__L[c] (touched 147 rxns)
  arg_L[c] -> arg__L[c] (touched 171 rxns)
  asn_L[c] -> asn__L[c] (touched 131 rxns)
  asp_L[c] -> asp__L[c] (touched 87 rxns)
  cys_L[c] -> cys__L[c] (touched 143 rxns)
  gln_L[c] -> gln__L[c] (touched 145 rxns)
  glu_L[c] -> glu__L[c] (touched 145 rxns)
  his_L[c] -> his__L[c] (touched 145 rxns)
  ile_L[c] -> ile__L[c] (touched 122 rxns)
  leu_L[c] -> leu__L[c] (touched 150 rxns)
  lys_L[c] -> lys__L[c] (touched 150 rxns)
  met_L[c] -> met__L[c] (touched 141 rxns)
  phe_L[c] -> phe__L[c] (touched 165 rxns)
  pro_L[c] -> pro__L[c] (touched 150 rxns)
  ser_L[c] -> ser__L[c] (touched 139 rxns)
  thr_L[c] -> thr__L[c] (touched 135 rxns)
  trp_L[c] -> trp__L[c] (touched 164 rxns)
  tyr_L[c] -> tyr__L[c] (touched 158 rxns)
  val_L[c] -> val__L[c] (touched 133 rxns)


In [19]:
# check blocked reactions 
block_final = cobra.flux_analysis.find_blocked_reactions(iMiceBrain)
len(block_final)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpyvpl49iv.lp
Reading time = 0.02 seconds
: 3131 rows, 12079 columns, 52737 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpk5taxsac.lp
Reading time = 0.02 seconds
: 3131 rows, 12079 columns, 52737 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpx_sdi9s3.lp
Reading time = 0.02 seconds
: 3131 rows, 12079 columns, 52737 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 

1

In [20]:
# add sink for apoe metabolite
from cobra import Reaction

met_id = "HC00009[c]"

# get the metabolite
met = iMiceBrain.metabolites.get_by_id(met_id)

# create sink reaction
sink_rxn = Reaction(f"SK_{met_id.replace('[','_').replace(']','')}")
sink_rxn.name = f"Sink of {met_id}"
sink_rxn.subsystem = "Exchange/demand reaction"
sink_rxn.lower_bound = 0.0     # only secretion
sink_rxn.upper_bound = 1000.0
sink_rxn.add_metabolites({met: -1.0})

# add to the model
iMiceBrain.add_reactions([sink_rxn])

print("Added:", sink_rxn.id, "for", met.id)

Added: SK_HC00009_c for HC00009[c]


In [21]:
# update model compartments 
iMiceBrain.compartments = {
    'r': 'Endoplasmic Reticulum',
    'c': 'Cytoplasm',
    'l': 'Lysosome',
    'm': 'Mitochondrion',
    'e': 'Extracellular',
    'g': 'Golgi apparatus',
    'x': 'Peroxisome',
    'n': 'Nucleus',
    'i': 'inner mitochondrial membrane'
}

In [22]:
len(iMiceBrain.genes), len(iMiceBrain.reactions), len(iMiceBrain.metabolites)

(1856, 6040, 3111)

In [23]:
# export the models after refining and then test through memote
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model
save_matlab_model(iMiceBrain, "/Users/eso1993/Desktop/iMiceBrain.mat")
write_sbml_model(iMiceBrain, "/Users/eso1993/Desktop/iMiceBrain.xml")

In [32]:
# check FVA 
import os
from cobra.flux_analysis import flux_variability_analysis
# Run FVA
FVA = flux_variability_analysis(iMiceBrain, fraction_of_optimum=1.0)

# Prepare metadata
reaction_ids = []
reaction_names = []
subsystems = []
genes = []

for rxn in iMiceBrain.reactions:
    reaction_ids.append(rxn.id)
    reaction_names.append(rxn.name)
    subsystems.append(rxn.subsystem)
    genes.append(";".join(g.id for g in rxn.genes))

# Create DataFrame
df = pd.DataFrame(FVA)
df.columns = ['minimum', 'maximum']
df['reaction_id'] = reaction_ids
df['reaction_name'] = reaction_names
df['subsystem'] = subsystems
df['genes'] = genes
df = df[['reaction_id', 'reaction_name', 'subsystem', 'genes', 'minimum', 'maximum']]

# Save to CSV
FVA_output = '/Users/eso1993/Desktop'
csv_filename = 'iMiceBrain_APOE_FVA.csv'
csv_file_path = os.path.join(FVA_output, csv_filename)
df.to_csv(csv_file_path, index=False)

Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpu2uvc74c.lp
Reading time = 0.02 seconds
: 5358 rows, 10265 columns, 45129 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpdba7wexn.lp
Reading time = 0.02 seconds
: 5358 rows, 10265 columns, 45129 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use only - expires 2025-11-14
Read LP format model from file /var/folders/24/08ydfvln4lq5k05zrf6n19mr0000gn/T/tmpfhb4vu3_.lp
Reading time = 0.02 seconds
: 5358 rows, 10265 columns, 45129 nonzeros
Set parameter Username
Set parameter LicenseID to value 2585127
Academic license - for non-commercial use 